# Visor 3D interactivo — ventana nativa VTK

Complemento del [PoC MVP 1](01-vtk-3dgs-poc.ipynb). Aquel renderiza *offscreen* a
**PNG** (fotos fijas, no se rotan). Este abre una **ventana interactiva** del
sistema donde puedes **rotar, hacer zoom y desplazar** el modelo con el ratón.

> ⚠️ **Requiere entorno gráfico (pantalla).** No funciona *headless* (servidor/CI)
> ni renderiza dentro del notebook: abre una **ventana aparte** del sistema
> operativo. La celda que la lanza **se queda bloqueada** hasta que **cierras la
> ventana** (tecla `q` o el botón de cerrar).
>
> **Cómo ejecutarlo:** desde la raíz del repo, `uv run jupyter notebook` (el
> prefijo `uv run` es lo que hace que el kernel encuentre `vtk`). Ejecuta las
> celdas en orden.

**Controles del ratón** (estilo *trackball*):

| Acción | Ratón |
|---|---|
| Rotar | Arrastrar con **botón izquierdo** |
| Zoom | **Rueda** (o arrastrar botón derecho) |
| Desplazar (pan) | **Shift + arrastrar**, o botón central |
| Cerrar la ventana | tecla **`q`** o cerrar la ventana |

Con el dataset **completo** en disco (300 pacientes / 600 escaneos), la §1 ya no
carga un caso fijo: **eliges cuál** —por paciente y arcada, o uno al azar— y las
§§3–5 lo abren en la ventana. Es la herramienta con la que se mira *de verdad* un
escaneo cuando un número del [notebook 01](01-vtk-3dgs-poc.ipynb) sorprende: por
qué esa arcada tiene 9 dientes, o por qué esa malla trae 25k vértices en vez de
116k.

No es el visor web «de producto» (eso es la Issue 3, three.js/GaussianSplats3D);
es una herramienta rápida de escritorio para inspeccionar el modelo real.

## 1 · Elegir el caso a inspeccionar

Mismo emparejamiento de los dos árboles que el [notebook 01](01-vtk-3dgs-poc.ipynb)
(un `.obj` sin su `.json` no cuenta como caso). Cambia `CASO` por cualquier
`(paciente, arcada)` de los 600 —o pon `CASO = caso_aleatorio()` para que te toque
uno al azar— y vuelve a ejecutar; las §§3–5 abrirán ese.


In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import vtk
from vtk.util.numpy_support import numpy_to_vtk

vtk.vtkObject.GlobalWarningDisplayOff()

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent
MESH_ROOT = ROOT / "data/raw/teeth3ds/3D_scans_per_patient_obj_files"
LABEL_ROOT = ROOT / "data/raw/teeth3ds/ground-truth_labels_instances"
assert MESH_ROOT.exists(), "Falta el dataset (corre scripts/fetch_teeth3ds.sh)"


def list_cases() -> list[tuple[str, str]]:
    """Casos (paciente, arcada) con malla Y labels."""
    return [
        (obj.stem.rsplit("_", 1)[0], obj.stem.rsplit("_", 1)[1])
        for d in sorted(p for p in MESH_ROOT.iterdir() if p.is_dir())
        for obj in sorted(d.glob("*.obj"))
        if (LABEL_ROOT / d.name / f"{obj.stem}.json").exists()
    ]


CASES = list_cases()


def caso_aleatorio(seed=None):
    return random.Random(seed).choice(CASES)


def cargar(pid, jaw):
    """Malla + etiquetas FDI pegadas como escalares de punto (para colorear)."""
    r = vtk.vtkOBJReader()
    r.SetFileName(str(MESH_ROOT / pid / f"{pid}_{jaw}.obj"))
    r.Update()
    p = r.GetOutput()
    lab = np.asarray(json.load(open(LABEL_ROOT / pid / f"{pid}_{jaw}.json"))["labels"], dtype=np.int32)
    assert len(lab) == p.GetNumberOfPoints(), "labels no casan con vértices"
    fa = numpy_to_vtk(lab.astype(float))
    fa.SetName("FDI")
    p.GetPointData().SetScalars(fa)
    return p, lab


# ---- EL CASO A MIRAR: cambialo por cualquiera de los 600, o usa caso_aleatorio()
CASO = ("01A6GW4A", "lower")
# CASO = caso_aleatorio()        # uno al azar
# CASO = caso_aleatorio(seed=3)  # uno al azar, pero siempre el mismo
# -----------------------------------------------------------------------------

PID, JAW = CASO
poly, labels = cargar(PID, JAW)
dientes = sorted(int(v) for v in np.unique(labels) if v != 0)
print(f"dataset: {len(CASES)} escaneos de {len({p for p, _ in CASES})} pacientes")
print(f"caso   : {PID}_{JAW} · {poly.GetNumberOfPoints():,} vértices · {len(dientes)} dientes")
print(f"FDI    : {dientes}")
print(f"encía  : {(labels == 0).mean() * 100:.0f}% de los vértices")


## 2 · El visor interactivo

La diferencia con el PoC 01 está en las **tres últimas líneas**: en vez de capturar a PNG, se crea un `vtkRenderWindowInteractor` con estilo *trackball* y se llama a `Start()`, que **cede el control al ratón** hasta que cierras la ventana.

In [ ]:
def view(actors, size=(900, 700), bg=(0.10, 0.10, 0.12), title="Teeth3DS · VTK interactivo"):
    # Abre una ventana nativa e interactiva. BLOQUEA la celda hasta que la cierras (tecla q).
    ren = vtk.vtkRenderer()
    for a in actors:
        ren.AddActor(a)
    ren.SetBackground(*bg)

    rw = vtk.vtkRenderWindow()          # ventana REAL (sin SetOffScreenRendering)
    rw.SetSize(*size)
    rw.SetWindowName(title)
    rw.AddRenderer(ren)

    iren = vtk.vtkRenderWindowInteractor()               # <-- el interactor
    iren.SetRenderWindow(rw)
    iren.SetInteractorStyle(vtk.vtkInteractorStyleTrackballCamera())  # rotar/zoom/pan

    ren.ResetCamera()
    rw.Render()
    iren.Start()                        # cede el control al ratón; bloquea la celda

## 3 · Rotar la malla coloreada por FDI

Al ejecutar se abre la ventana. **Arrastra con el ratón para girar.** Ciérrala (tecla `q`) para que la celda termine y puedas seguir.

In [ ]:
m = vtk.vtkPolyDataMapper()
m.SetInputData(poly)
m.SetScalarRange(0, 47)
mesh_actor = vtk.vtkActor()
mesh_actor.SetMapper(m)

view([mesh_actor], title=f"{PID}_{JAW} · malla por FDI")

## 4 · Rotar el campo `vtkGaussianSplatter` (opcional)

Mismo campo de densidad que el PoC 01 (§4), pero rotable. Útil para ver de qué lado el *splatting* reconstruye bien y de cuál queda disperso.

In [ ]:
cloud = vtk.vtkPolyData()
cloud.SetPoints(poly.GetPoints())

splat = vtk.vtkGaussianSplatter()
splat.SetInputData(cloud)
splat.SetSampleDimensions(80, 80, 80)
splat.SetRadius(0.025)
splat.ScalarWarpingOff()
splat.Update()

iso = vtk.vtkContourFilter()
iso.SetInputData(splat.GetOutput())
iso.SetValue(0, 0.15)
iso.Update()
im = vtk.vtkPolyDataMapper()
im.SetInputConnection(iso.GetOutputPort())
im.ScalarVisibilityOff()
splat_actor = vtk.vtkActor()
splat_actor.SetMapper(im)
splat_actor.GetProperty().SetColor(0.90, 0.85, 0.75)

view([splat_actor], title=f"{PID}_{JAW} · campo gaussiano")

## 5 · Rotar la nube de puntos (vértices)

Los **vértices sueltos** de la malla — los «ladrillos» que alimentan al
`vtkGaussianSplatter` (§4) — pero aquí **rotables**. Útil para ver la densidad y
distribución real de los puntos: dónde el escaneo tiene detalle y dónde queda ralo.
Se colorean por FDI, igual que la malla.

In [ ]:
# Nube de puntos = solo los vértices, sin caras. vtkVertexGlyphFilter los hace dibujables.
vg = vtk.vtkVertexGlyphFilter()
vg.SetInputData(poly)          # poly ya trae los escalares FDI de la carga (§1)
vg.Update()

pm = vtk.vtkPolyDataMapper()
pm.SetInputConnection(vg.GetOutputPort())
pm.SetScalarRange(0, 47)       # colorear por FDI (encía 0 … dientes)
points_actor = vtk.vtkActor()
points_actor.SetMapper(pm)
points_actor.GetProperty().SetPointSize(2)

view([points_actor], title=f"{PID}_{JAW} · nube de puntos (vértices)")